Created Train/Test split with the latest month (September 2026) being the test split and the two years before (September 2024 through August 2026) being the train split.

In [1]:
from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
CSV_DIR = DATA_DIR / "raw/csv"

from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
CSV_DIR = DATA_DIR / "raw/csv"

TEST_MONTH = "202609"
TRAIN_MONTHS = [
    "202409", "202410", "202411", "202412",
    "202501", "202502", "202503", "202504",
    "202505", "202506", "202507", "202508"
]

In [2]:
def sold_path(month):
    candidates = [
        CSV_DIR / f"GeneratedCRMLSSold{month}.csv",
        CSV_DIR / f"CRMLSSold{month}.csv",
        CSV_DIR / f"CRMLSSold{month}_filled.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"No sold CSV found for month {month}")


def load_month(month):
    df = pd.read_csv(sold_path(month), low_memory=False)
    df["month"] = month
    return df


train = pd.concat([load_month(m) for m in TRAIN_MONTHS], ignore_index=True)
test = load_month(TEST_MONTH)

print("train months", TRAIN_MONTHS)
print(train["month"].value_counts().sort_index())
print(test["month"].value_counts().sort_index())

train months ['202409', '202410', '202411', '202412', '202501', '202502', '202503', '202504', '202505', '202506', '202507', '202508']
month
202409    21267
202410    23274
202411    20279
202412    20241
202501    18738
202502    18702
202503    21445
202504    23262
202505    23154
202506    22883
202507    23646
202508    22972
Name: count, dtype: int64
month
202609    22107
Name: count, dtype: int64


Combining test/train tables into one in order to handle null values and encode categorical columns. 

In [3]:
df = pd.concat(
    [train.assign(split="train"), test.assign(split="test")],
    ignore_index=True,
)
df["split"].value_counts()

split
train    259863
test      22107
Name: count, dtype: int64

Filter to residential single-family homes in California, then drop StateOrProvince and PropertyType since they are constant after the filter.

In [4]:
df = df[
    (df["PropertyType"] == "Residential")
    & (df["PropertySubType"] == "SingleFamilyResidence")
    & (df["StateOrProvince"] == "CA")
]
df = df.drop(columns=["StateOrProvince", "PropertyType"])
df["split"].value_counts()

split
train    131093
test      11191
Name: count, dtype: int64

Keep the home features used for modeling, columns relating to listing itself are not included

In [5]:
keep_cols = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "PropertySubType",
    "WaterfrontYN",
    "BasementYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "PoolPrivateYN",
    "ViewYN",
    "NewConstructionYN",
    "City",
    "PostalCode",
    "Latitude",
    "Longitude",
    "month",
    "Flooring",
    "FireplacesTotal",
    "AssociationFeeFrequency",
    "CountyOrParish",
    "ParkingTotal",
    "LotSizeAcres",
    "BuildingAreaTotal",
    "ElementarySchoolDistrict",
    "CoveredSpaces",
    "Stories",
    "Levels",
    "HighSchoolDistrict",
    "AssociationFee",
    "MiddleOrJuniorSchoolDistrict",
    "split",
] 
df = df[keep_cols]

Remove outliers on ClosePrice and other numeric home features. Bounds are the train-set 1st and 99th percentiles (fit on train only to avoid leakage), then applied to both splits. Nulls in feature columns are kept for later imputation.

In [6]:
outlier_cols = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "LotSizeAcres",
    "BuildingAreaTotal",
    "ParkingTotal",
    "Stories",
    "AssociationFee",
]

train_mask = df["split"] == "train"
n_before = len(df)
keep = pd.Series(True, index=df.index)

for col in outlier_cols:
    if col not in df.columns:
        continue
    lo = df.loc[train_mask, col].quantile(0.01)
    hi = df.loc[train_mask, col].quantile(0.99)
    if col == "ClosePrice":
        keep &= df[col].between(lo, hi)
    else:
        # keep missing values; only drop extreme observed values
        keep &= df[col].isna() | df[col].between(lo, hi)
    print(f"{col}: keep [{lo:.4g}, {hi:.4g}]")

df = df[keep].copy()
print(f"rows {n_before} -> {len(df)} (removed {n_before - len(df)})")
df["split"].value_counts()

ClosePrice: keep [2.4e+05, 6.342e+06]
LivingArea: keep [728, 5647]
BedroomsTotal: keep [2, 6]
BathroomsTotalInteger: keep [1, 6]
LotSizeSquareFeet: keep [1800, 2.413e+05]
YearBuilt: keep [1910, 2025]
GarageSpaces: keep [0, 4]
LotSizeAcres: keep [0.0416, 5.597]
BuildingAreaTotal: keep [649.5, 8998]
ParkingTotal: keep [0, 11]
Stories: keep [1, 2]
AssociationFee: keep [0, 1000]
rows 142284 -> 130240 (removed 12044)


split
train    120379
test       9861
Name: count, dtype: int64

Determining the proportion of entries that have a null value for each column

In [7]:
null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
null_pct

MiddleOrJuniorSchoolDistrict    100.000000
CoveredSpaces                   100.000000
FireplacesTotal                 100.000000
ElementarySchoolDistrict        100.000000
WaterfrontYN                     99.971591
BasementYN                       97.848587
BuildingAreaTotal                93.886671
AssociationFeeFrequency          75.556665
Flooring                         35.428440
AssociationFee                   30.406941
HighSchoolDistrict               25.928286
Stories                          11.947174
AttachedGarageYN                 10.425369
ViewYN                            9.398034
PoolPrivateYN                     9.133907
Levels                            9.101658
NewConstructionYN                 7.245086
GarageSpaces                      3.547297
LotSizeSquareFeet                 1.741400
LotSizeAcres                      1.741400
City                              0.075246
FireplaceYN                       0.049140
YearBuilt                         0.043765
LivingArea 

Drop columns that are null for every row, observe remaining columns

In [8]:
all_null = df.columns[df.isna().all()]
df = df.drop(columns=all_null)

# Dropping null Close Price rows because they are not useful for modeling,
# Dropping null Latitude and Longitude rows because of low null percentages
df = df.dropna(subset=["ClosePrice", "Latitude", "Longitude"]) 

null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print(null_pct.to_string())


WaterfrontYN               99.971582
BasementYN                 97.847926
BuildingAreaTotal          93.884793
AssociationFeeFrequency    75.555300
Flooring                   35.432412
AssociationFee             30.391705
HighSchoolDistrict         25.918587
Stories                    11.940092
AttachedGarageYN           10.426267
ViewYN                      9.388633
PoolPrivateYN               9.125960
Levels                      9.093702
NewConstructionYN           7.247312
GarageSpaces                3.548387
LotSizeAcres                1.741167
LotSizeSquareFeet           1.741167
City                        0.075269
FireplaceYN                 0.049155
YearBuilt                   0.043779
LivingArea                  0.033794
BathroomsTotalInteger       0.016129
ParkingTotal                0.001536
PostalCode                  0.000768


See the values a column takes, including null

For waterfront column, notice that the only non-null value is true meaning we can impute the null values to be false and then encode the columns by making true = 1 and false = 0

In [9]:
def colValues(col):
    counts = df[col].value_counts(dropna=False)
    print(f"{len(counts)} distinct values")
    print(counts.to_string())

colValues("WaterfrontYN")

2 distinct values
WaterfrontYN
NaN     130163
True        37


In [10]:
df["WaterfrontYN"] = df["WaterfrontYN"].eq(True).astype(int)
df["BasementYN"] = df["BasementYN"].eq(True).astype(int)

For LotSize, avg sqft col and acres (in sqft) col, if one is null take the other measurement. If both are null flag in the missing column and set to 0. 

In [11]:
sqft = df["LotSizeSquareFeet"]
acres_sqft = df["LotSizeAcres"] * 43560
lot = (sqft.fillna(acres_sqft) + acres_sqft.fillna(sqft)) / 2

df["LotSizeSquareFeet_missing"] = lot.isna().astype(int)
df["LotSizeSquareFeet"] = lot.fillna(0)
df = df.drop(columns=["LotSizeAcres"])

For other YN columns, 1 for True, 0 for False, and 0 for null and flag in a col_missing column where 0 is not missing and 1 is missing

Convert Levels strings to a numeric story count (take the max if multiple values are listed). Convert AssociationFee to a monthly amount using AssociationFeeFrequency (Annual → /12, SemiAnnually → /6, Quarterly → /3, Monthly → unchanged), then drop the original fee columns.

Flag other columns missing that have high null percentages

In [12]:
yn_cols = [
    "AttachedGarageYN",
    "FireplaceYN",
    "PoolPrivateYN",
    "ViewYN",
    "NewConstructionYN",
]

for col in yn_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].map({True: 1, False: 0}).fillna(0).astype(int)

level_map = {
    "One": 1,
    "Two": 2,
    "ThreeOrMore": 3,
    "MultiSplit": 1.5,
}

def levels_to_numeric(value):
    if pd.isna(value):
        return pd.NA
    nums = [level_map[part.strip()] for part in str(value).split(",") if part.strip() in level_map]
    return max(nums) if nums else pd.NA

df["Levels_missing"] = df["Levels"].isna().astype(int)
df["Levels"] = df["Levels"].map(levels_to_numeric).astype("Float64").fillna(0)

freq_to_monthly = {
    "Monthly": 1,
    "Quarterly": 1 / 3,
    "SemiAnnually": 1 / 6,
    "Annually": 1 / 12,
}
monthly_fee = df["AssociationFee"] * df["AssociationFeeFrequency"].map(freq_to_monthly)
df["AssociationFeeMonthly_missing"] = monthly_fee.isna().astype(int)
df["AssociationFeeMonthly"] = monthly_fee.fillna(0)
df = df.drop(columns=["AssociationFee", "AssociationFeeFrequency"])

flag_cols = [
    "GarageSpaces",
    "Stories",
    "BuildingAreaTotal",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "ParkingTotal",
]

for col in flag_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(0)

df.head(2)

,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,PropertySubType,WaterfrontYN,BasementYN,...,AssociationFeeMonthly_missing,AssociationFeeMonthly,GarageSpaces_missing,Stories_missing,BuildingAreaTotal_missing,LivingArea_missing,BedroomsTotal_missing,BathroomsTotalInteger_missing,YearBuilt_missing,ParkingTotal_missing
0,1090000.0,2655.0,3.0,4.0,14388.934,1971.0,0.0,SingleFamilyResidence,0,0,...,1,0.0,0,0,1,0,0,0,0,0
7,1995000.0,3524.0,4.0,5.0,18295.100,2004.0,3.0,SingleFamilyResidence,0,0,...,0,850.0,0,0,1,0,0,0,0,0


Encode month as a numeric index (0 = first train month, then +1 each month through the test month).

Target-encode City, PostalCode, and HighSchoolDistrict using the train-set mean ClosePrice, with additive smoothing so rare categories shrink toward the global train mean. Means are fit on train only; unseen or missing values get the global train mean. Low-cardinality columns (PropertySubType, CountyOrParish) stay one-hot encoded.

In [13]:
month_order = TRAIN_MONTHS + [TEST_MONTH]
month_index = {m: i for i, m in enumerate(month_order)}
df["month"] = df["month"].map(month_index).astype(int)

df["PostalCode"] = df["PostalCode"].astype("string").str.replace(r"\.0$", "", regex=True)

train = df["split"] == "train"
global_mean = df.loc[train, "ClosePrice"].mean()
smooth_k = 20

target_cols = ["City", "PostalCode", "HighSchoolDistrict"]
for col in target_cols:
    keys = df[col].astype("string").fillna("__missing__")
    df[f"{col}_missing"] = df[col].isna().astype("int8")
    stats = df.loc[train].groupby(keys[train], dropna=False)["ClosePrice"].agg(["mean", "count"])
    encoded_mean = (stats["count"] * stats["mean"] + smooth_k * global_mean) / (stats["count"] + smooth_k)
    df[f"{col}_te"] = keys.map(encoded_mean).fillna(global_mean)
    df = df.drop(columns=[col])

materials = sorted({part.strip() for value in df["Flooring"].dropna() for part in str(value).split(",") if part.strip()})
flooring_missing = df["Flooring"].isna()
flooring = df["Flooring"].fillna("")
for material in materials:
    df[f"Flooring_{material}"] = flooring.str.contains(rf"(?:^|,){material}(?:,|$)", regex=True).astype("int8")
df["Flooring_missing"] = flooring_missing.astype("int8")
df = df.drop(columns=["Flooring"])

cat_cols = ["PropertySubType", "CountyOrParish"]
df = pd.get_dummies(df, columns=cat_cols, dummy_na=True, dtype="int8")
df.shape

(130200, 116)

Now, all of the variables should be numerically encoded with no null values present in the dataframe. Also check that no categorical/object columns remain.

In [14]:
null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print(null_pct.to_string())

Series([], )


In [15]:
cat_remaining = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
print("remaining categorical columns:", cat_remaining)
print("dtypes:\n", df.dtypes.value_counts().to_string())

remaining categorical columns: ['split']
dtypes:
 int8       74
int64      24
float64    16
Float64     1
str         1


Save the target-encoded dataframe for the modeling notebook.

In [16]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "sales_preprocessed_1_target_encoded.csv"
df.to_csv(out_path, index=False)